# Pigmentation segmentation, v2 (Mask R-CNN, Colab GPU)

What changed vs. the first notebook:

1. **LR schedule fixed**: warmup then cosine decay (old `StepLR` was down to ~1e-7 by epoch 10).
2. **Augmentation**: flips, rot90, colour jitter, random CLAHE, multi-scale training.
3. **Train/inference preprocessing now match**: CLAHE is used as a training augmentation, and we *measure* whether it helps on validation.
4. **Data checks**: prints classes in each `data.yaml` / label files, optional class filter, and a **leakage-safe split** (grouped by source image, so Roboflow augmented copies never straddle train/val).
5. **Model tweaks**: smaller anchors for small spots, optional `maskrcnn_resnet50_fpn_v2`.
6. **Better evaluation**: instance precision/recall/F1 (false positives count), pixel Dice/IoU, COCO mask mAP, score-threshold sweep, optional flip TTA.
7. **Checkpoints actually used**: resume after Colab disconnects, save best epoch, early stopping.

Run top to bottom. Read the notes in sections 3 and 4 before training.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q torchmetrics pycocotools opencv-python-headless pyyaml

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

# New folder name on purpose, so old checkpoints from the v1 notebook are never loaded by accident
CHECKPOINT_DIR = '/content/drive/MyDrive/pigmentation_maskrcnn_v2_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
LATEST_PATH = os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pth')
BEST_PATH   = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
print('Checkpoints ->', CHECKPOINT_DIR)

## 1. Config
Everything you might want to tune is here. If you set `USE_V2 = True`, also raise `BATCH_SIZE` to 8+ (the v2 model uses regular BatchNorm, which needs bigger batches; v1 uses frozen BatchNorm and is fine at 4).

In [ ]:
# ===================== CONFIG =====================
SEED = 42

# training
NUM_EPOCHS    = 40
BATCH_SIZE    = 4
LR            = 1e-4        # AdamW
WEIGHT_DECAY  = 1e-4
WARMUP_EPOCHS = 1
MIN_LR_RATIO  = 0.01        # final LR = LR * this
PATIENCE      = 10          # early stop if val F1 doesn't improve for this many epochs
RESUME        = True        # resume from latest_checkpoint.pth if it exists (set False for a fresh run)

# model
USE_V2       = False        # True -> maskrcnn_resnet50_fpn_v2 (use BATCH_SIZE >= 8)
ANCHOR_SIZES = (16, 32, 64, 128, 256)         # default is (32..512); smaller = better for tiny spots
MIN_SIZES    = (640, 672, 704, 736, 768, 800) # multi-scale training; eval uses the last one (800)
MAX_SIZE     = 1333

# augmentation
CLAHE_TRAIN_PROB   = 0.5    # train sees both raw and CLAHE images
ALLOW_ROT90_VFLIP  = True   # set False if you only care about upright faces

# evaluation
EVAL_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]   # score thresholds swept during training
MATCH_IOU       = 0.5                                # IoU needed to count a predicted spot as correct

# datasets
ZIP_FILES = [
    '/content/roboflow1.v1i.yolov11.zip',
    '/content/spots redness pigmentation.v1i.yolov11.zip',
]
EXTRACT_DIR = '/content/pigmentation_datasets'
# ==================================================

In [ ]:
import os, re, json, math, random, shutil, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import cv2
import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ColorJitter
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

from torchvision.models.detection import (
    maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights,
    maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.anchor_utils import AnchorGenerator

assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU'
device = torch.device('cuda')
torch.backends.cudnn.benchmark = True
cv2.setNumThreads(0)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

print('torch', torch.__version__, '| torchvision', torchvision.__version__, '|', torch.cuda.get_device_name(0))

## 2. Extract the datasets
We no longer copy everything into one folder (that threw away which split and which source image each file came from).

In [ ]:
for z in ZIP_FILES:
    print(z, '->', 'FOUND' if os.path.exists(z) else 'NOT FOUND')

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR)
for i, zp in enumerate(ZIP_FILES, start=1):
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(os.path.join(EXTRACT_DIR, f'dataset_{i}'))
print('Extracted.')

## 3. CHECK YOUR CLASSES (important)
The old notebook read the class id from every label line and then **ignored it**, so every polygon became "pigmentation". If the second dataset has classes like `spots` / `redness` / `pigmentation`, those were all merged. Run this cell and look at the output, then set `KEEP_CLASSES` in the next cell.

In [ ]:
import yaml
for folder in sorted(Path(EXTRACT_DIR).glob('dataset_*')):
    ycfg = folder / 'data.yaml'
    if ycfg.exists():
        cfg = yaml.safe_load(open(ycfg))
        print(folder.name, '| nc =', cfg.get('nc'), '| names =', cfg.get('names'))
    counts = Counter()
    for lf in folder.glob('*/labels/*.txt'):
        for line in open(lf):
            parts = line.split()
            if parts:
                counts[int(float(parts[0]))] += 1
    print('   polygons per class id in label files:', dict(sorted(counts.items())))

In [ ]:
# Which class ids to keep from each dataset (None = keep all).
# Example: if dataset_2 has 0=redness, 1=pigmentation, 2=spots and you only want pigmentation + spots:
#   KEEP_CLASSES = {1: None, 2: {1, 2}}
KEEP_CLASSES = {1: None, 2: None}

## 4. Leakage-safe train/val split
Your roboflow1 export has 960 images but only ~657 distinct source photos (I checked), and 54 of those photos appear in more than one split. A plain random split would put copies of one photo in both train and val.

Roboflow exports contain augmented copies of the same photo (`1003_image-111_jpg.rf.<hash>.jpg`, `1004_image-111_jpg.rf.<hash>.jpg`). A plain random split puts copies of one photo in both train and val, which inflates validation scores and hides real performance. Here we group by source photo and split by group.

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

all_samples = []
for folder in sorted(Path(EXTRACT_DIR).glob('dataset_*')):
    ds = int(folder.name.split('_')[1])
    for split in ['train', 'valid', 'test']:
        imgs, lbls = folder / split / 'images', folder / split / 'labels'
        if not imgs.exists() or not lbls.exists():
            continue
        for p in sorted(imgs.iterdir()):
            if p.suffix.lower() not in IMG_EXT:
                continue
            lp = lbls / f'{p.stem}.txt'
            if not lp.exists():
                continue
            # strip Roboflow's augmentation hash AND the leading counter ("1003_image-111_jpg" and
            # "1004_image-111_jpg" are two augmented copies of the same photo)
            src = re.sub(r'^\d+_', '', p.stem.split('.rf.')[0])
            group = f'{ds}_{src}'
            all_samples.append(dict(image=str(p), label=str(lp), ds=ds, group=group))

groups = [s['group'] for s in all_samples]
print('Images:', len(all_samples), '| unique source photos:', len(set(groups)))

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr_idx, va_idx = next(gss.split(all_samples, groups=groups))
train_samples = [all_samples[i] for i in tr_idx]
val_samples   = [all_samples[i] for i in va_idx]

assert not ({s['group'] for s in train_samples} & {s['group'] for s in val_samples}), 'leakage!'
print('Train:', len(train_samples), '| Val:', len(val_samples), '(no shared source photos)')

## 5. Dataset with augmentation
- Train: colour jitter, random CLAHE, flips (+ vflip/rot90 if allowed). Masks are transformed together with the image and boxes are recomputed from the masks.
- Val: no randomness. CLAHE is applied to every val image only if `clahe_prob=1.0` (we compare both at the end).

In [ ]:
def apply_clahe(rgb):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)


def random_geometry(img, masks):
    # img: HxWx3 uint8, masks: NxHxW uint8
    if random.random() < 0.5:
        img, masks = img[:, ::-1], masks[:, :, ::-1]
    if ALLOW_ROT90_VFLIP:
        if random.random() < 0.5:
            img, masks = img[::-1], masks[:, ::-1]
        k = random.randint(0, 3)
        if k:
            img, masks = np.rot90(img, k, (0, 1)), np.rot90(masks, k, (1, 2))
    return np.ascontiguousarray(img), np.ascontiguousarray(masks)


class PigmentationDataset(Dataset):
    def __init__(self, samples, train=False, clahe_prob=0.0):
        self.samples = samples
        self.train = train
        self.clahe_prob = clahe_prob
        self.jitter = ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.02)

    def __len__(self):
        return len(self.samples)

    def _polygons(self, sample, w, h):
        keep = KEEP_CLASSES.get(sample['ds'])
        polys = []
        with open(sample['label']) as f:
            for line in f:
                v = line.split()
                if len(v) < 7:                      # class id + at least 3 points
                    continue
                cid = int(float(v[0]))
                if keep is not None and cid not in keep:
                    continue
                c = list(map(float, v[1:]))
                if len(c) % 2:
                    c = c[:-1]
                polys.append([(c[i] * w, c[i + 1] * h) for i in range(0, len(c), 2)])
        return polys

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s['image']).convert('RGB')
        w, h = img.size

        masks = []
        for poly in self._polygons(s, w, h):
            m = Image.new('L', (w, h), 0)
            ImageDraw.Draw(m).polygon(poly, outline=1, fill=1)
            masks.append(np.array(m, dtype=np.uint8))
        masks = np.stack(masks) if masks else np.zeros((0, h, w), np.uint8)

        if self.train:
            img = self.jitter(img)
        img = np.array(img)
        if random.random() < self.clahe_prob:
            img = apply_clahe(img)
        if self.train:
            img, masks = random_geometry(img, masks)

        boxes, kept = [], []
        for i, m in enumerate(masks):
            ys, xs = np.where(m > 0)
            if len(xs) < 4:                         # drop empty / tiny masks
                continue
            boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
            kept.append(i)
        H, W = img.shape[:2]
        if kept:
            masks_t = torch.from_numpy(masks[kept].copy())
            boxes_t = torch.as_tensor(boxes, dtype=torch.float32)
        else:
            masks_t = torch.zeros((0, H, W), dtype=torch.uint8)
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)

        img_t = torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1).float() / 255.0
        target = dict(boxes=boxes_t,
                      labels=torch.ones((len(boxes_t),), dtype=torch.int64),
                      masks=masks_t,
                      image_id=torch.tensor([idx]))
        return img_t, target


def collate_fn(batch):
    return tuple(zip(*batch))

train_ds = PigmentationDataset(train_samples, train=True, clahe_prob=CLAHE_TRAIN_PROB)
print('Train images:', len(train_ds))

In [ ]:
# Visual check: the SAME training image augmented 4 times. Masks must stay glued to the spots.
i = next((k for k in range(len(train_ds)) if len(train_ds[k][1]['masks']) > 0), 0)
fig, axs = plt.subplots(1, 4, figsize=(20, 5))
for ax in axs:
    im, t = train_ds[i]
    ax.imshow(im.permute(1, 2, 0).numpy())
    for m in t['masks']:
        ax.contour(m.numpy(), levels=[0.5], colors='red', linewidths=1)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
def make_val_loader(clahe):
    ds = PigmentationDataset(val_samples, train=False, clahe_prob=1.0 if clahe else 0.0)
    return ds, DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                          pin_memory=True, collate_fn=collate_fn)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                          pin_memory=True, collate_fn=collate_fn, drop_last=True,
                          persistent_workers=True)

# During training we validate on raw (non-CLAHE) images; the final section compares both.
val_ds, val_loader = make_val_loader(clahe=False)
print('train batches:', len(train_loader), '| val batches:', len(val_loader))

## 6. Model
- Default: `maskrcnn_resnet50_fpn` (COCO weights, frozen BatchNorm, all backbone layers trainable).
- Smaller anchors (16 to 256) so tiny spots get proposals. The anchor count per location stays 3, so pretrained RPN weights still load; they just re-adapt.

In [ ]:
def build_model(pretrained=True, use_v2=None, anchor_sizes=None, num_classes=2):
    use_v2 = USE_V2 if use_v2 is None else use_v2
    anchor_sizes = ANCHOR_SIZES if anchor_sizes is None else anchor_sizes
    kw = dict(min_size=MIN_SIZES, max_size=MAX_SIZE)
    if pretrained:
        kw['trainable_backbone_layers'] = 5
    if use_v2:
        w = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None
        model = maskrcnn_resnet50_fpn_v2(weights=w, weights_backbone=None, **kw)
    else:
        w = MaskRCNN_ResNet50_FPN_Weights.DEFAULT if pretrained else None
        model = maskrcnn_resnet50_fpn(weights=w, weights_backbone=None, **kw)

    model.rpn.anchor_generator = AnchorGenerator(
        sizes=tuple((s,) for s in anchor_sizes),
        aspect_ratios=((0.5, 1.0, 2.0),) * len(anchor_sizes))

    in_f = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_f, num_classes)
    in_m = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_m, 256, num_classes)
    return model

model = build_model(pretrained=True).to(device)
print('Model:', 'Mask R-CNN R50-FPN', 'v2' if USE_V2 else 'v1',
      '| trainable params: %.1fM' % (sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6))

## 7. Optimizer + per-iteration warmup/cosine schedule

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

steps_per_epoch = len(train_loader)
total_steps  = NUM_EPOCHS * steps_per_epoch
warmup_steps = WARMUP_EPOCHS * steps_per_epoch

def lr_lambda(step):
    if step < warmup_steps:
        return 0.1 + 0.9 * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * progress))

lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler('cuda')

# quick look at the schedule
xs = list(range(0, total_steps, max(1, total_steps // 200)))
plt.figure(figsize=(6, 2.5)); plt.plot([x / steps_per_epoch for x in xs], [LR * lr_lambda(x) for x in xs])
plt.xlabel('epoch'); plt.ylabel('LR'); plt.title('LR schedule'); plt.show()

## 8. Evaluation tools
- `evaluate`: for every score threshold: instance precision / recall / **F1** (a predicted spot is correct if IoU ≥ 0.5 with an unused ground-truth spot, so false positives are penalised), plus pixel-level Dice/IoU of "all predicted spots vs all true spots".
- `coco_map`: standard COCO mask mAP (run at the end).
- `predict(..., tta=True)`: averages in a horizontally flipped pass (merged with NMS).

In [ ]:
def predict(model, imgs, tta=False):
    preds = model(imgs)
    if not tta:
        return preds
    preds_f = model([im.flip(-1) for im in imgs])
    out = []
    for im, p, pf in zip(imgs, preds, preds_f):
        W = im.shape[-1]
        bf = pf['boxes'].clone()
        bf[:, [0, 2]] = W - pf['boxes'][:, [2, 0]]
        boxes  = torch.cat([p['boxes'], bf])
        scores = torch.cat([p['scores'], pf['scores']])
        labels = torch.cat([p['labels'], pf['labels']])
        masks  = torch.cat([p['masks'], pf['masks'].flip(-1)])
        k = torchvision.ops.nms(boxes, scores, 0.5)
        out.append(dict(boxes=boxes[k], scores=scores[k], labels=labels[k], masks=masks[k]))
    return out


@torch.no_grad()
def evaluate(model, loader, thresholds, tta=False, match_iou=MATCH_IOU):
    model.eval()
    thresholds = list(thresholds)
    acc = {t: dict(tp=0, fp=0, fn=0, inter=0.0, psum=0.0, gsum=0.0) for t in thresholds}
    min_t = min(thresholds)

    for images, targets in loader:
        imgs = [im.to(device) for im in images]
        with torch.autocast('cuda'):
            preds = predict(model, imgs, tta)

        for p, t in zip(preds, targets):
            gt = t['masks'].to(device).bool()                  # [G,H,W]
            G = len(gt)
            H, W = gt.shape[-2:]
            keep = p['scores'] >= min_t
            scores = p['scores'][keep].float()
            pm = p['masks'][keep][:, 0] > 0.5                  # [P,H,W]
            gt_flat = gt.flatten(1).float()
            gt_area = gt_flat.sum(1)
            gu = gt.any(0) if G else torch.zeros(H, W, dtype=torch.bool, device=device)

            for thr in thresholds:
                pmt = pm[scores >= thr]
                P = len(pmt)
                a = acc[thr]
                tp = 0
                if P and G:
                    pf = pmt.flatten(1).float()
                    inter = pf @ gt_flat.T
                    union = pf.sum(1)[:, None] + gt_area[None] - inter
                    iou = (inter / union.clamp(min=1)).cpu().numpy()
                    used = set()
                    for pi in range(P):                        # preds are sorted by score
                        for gi in iou[pi].argsort()[::-1]:
                            if iou[pi, gi] < match_iou:
                                break
                            if gi not in used:
                                used.add(gi); tp += 1
                                break
                a['tp'] += tp; a['fp'] += P - tp; a['fn'] += G - tp
                pu = pmt.any(0) if P else torch.zeros(H, W, dtype=torch.bool, device=device)
                a['inter'] += (pu & gu).sum().item()
                a['psum']  += pu.sum().item()
                a['gsum']  += gu.sum().item()

    rows = {}
    for t, a in acc.items():
        tp, fp, fn = a['tp'], a['fp'], a['fn']
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec  = tp / (tp + fn) if tp + fn else 0.0
        f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        den  = a['psum'] + a['gsum']
        dice = 2 * a['inter'] / den if den else 1.0
        uni  = den - a['inter']
        iou  = a['inter'] / uni if uni else 1.0
        rows[t] = dict(precision=prec, recall=rec, f1=f1, dice=dice, iou=iou)
    return rows


@torch.no_grad()
def coco_map(model, loader, tta=False):
    from torchmetrics.detection import MeanAveragePrecision
    metric = MeanAveragePrecision(iou_type='segm')
    model.eval()
    for images, targets in loader:
        imgs = [im.to(device) for im in images]
        with torch.autocast('cuda'):
            preds = predict(model, imgs, tta)
        pl, tl = [], []
        for p, t in zip(preds, targets):
            k = p['scores'] >= 0.05
            pl.append(dict(scores=p['scores'][k].float().cpu(), labels=p['labels'][k].cpu(),
                           masks=(p['masks'][k][:, 0] > 0.5).cpu()))
            tl.append(dict(labels=t['labels'], masks=t['masks'].bool()))
        metric.update(pl, tl)
    res = metric.compute()
    return {k: float(v) for k, v in res.items() if v.numel() == 1}


def best_row(rows):
    t = max(rows, key=lambda k: rows[k]['f1'])
    return t, rows[t]

## 9. Train
Resumes automatically after a Colab disconnect (model, optimizer, scheduler, scaler, history). Saves `best_model.pth` whenever validation F1 improves and stops early if it stalls.

In [ ]:
CONFIG_SNAPSHOT = dict(use_v2=USE_V2, anchor_sizes=list(ANCHOR_SIZES), min_sizes=list(MIN_SIZES),
                       max_size=MAX_SIZE, num_classes=2)

history = dict(train_loss=[], val_f1=[], val_dice=[], val_thr=[], lr=[])
start_epoch, best_f1, bad_epochs = 0, -1.0, 0

if RESUME and os.path.exists(LATEST_PATH):
    ck = torch.load(LATEST_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    lr_scheduler.load_state_dict(ck['scheduler'])
    scaler.load_state_dict(ck['scaler'])
    history, best_f1, bad_epochs = ck['history'], ck['best_f1'], ck['bad_epochs']
    start_epoch = ck['epoch'] + 1
    print(f'Resumed. Next epoch: {start_epoch + 1}/{NUM_EPOCHS} | best F1 so far: {best_f1:.4f}')
else:
    print('Starting fresh.')

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    running, n_ok = 0.0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{NUM_EPOCHS}')
    for images, targets in pbar:
        images  = [im.to(device, non_blocking=True) for im in images]
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda'):
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

        if not torch.isfinite(loss):
            print('Non-finite loss, skipping batch:', {k: float(v) for k, v in loss_dict.items()})
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(params, 5.0)
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()

        running += loss.item(); n_ok += 1
        pbar.set_postfix(loss=f'{loss.item():.3f}', avg=f'{running / max(1, n_ok):.3f}')

    avg_loss = running / max(1, n_ok)
    rows = evaluate(model, val_loader, EVAL_THRESHOLDS)
    thr, r = best_row(rows)

    history['train_loss'].append(avg_loss); history['val_f1'].append(r['f1'])
    history['val_dice'].append(r['dice']);  history['val_thr'].append(thr)
    history['lr'].append(optimizer.param_groups[0]['lr'])
    print(f"Epoch {epoch + 1}: loss {avg_loss:.4f} | val F1 {r['f1']:.4f} (P {r['precision']:.3f} / R {r['recall']:.3f}) "
          f"@thr {thr} | pixel Dice {r['dice']:.4f} IoU {r['iou']:.4f} | lr {history['lr'][-1]:.2e}")

    improved = r['f1'] > best_f1 + 1e-4
    if improved:
        best_f1, bad_epochs = float(r['f1']), 0
        torch.save(dict(model=model.state_dict(), config=CONFIG_SNAPSHOT,
                        threshold=float(thr), val_f1=best_f1, epoch=epoch), BEST_PATH)
        print('   new best model saved')
    else:
        bad_epochs += 1

    torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                    scheduler=lr_scheduler.state_dict(), scaler=scaler.state_dict(),
                    history=history, best_f1=best_f1, bad_epochs=bad_epochs, epoch=epoch), LATEST_PATH)

    if bad_epochs >= PATIENCE:
        print(f'Early stopping: no improvement for {PATIENCE} epochs.')
        break

print('Done. Best val F1:', round(best_f1, 4))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(history['train_loss']); axs[0].set_title('Train loss'); axs[0].set_xlabel('epoch')
axs[1].plot(history['val_f1'], label='val F1 (instances)')
axs[1].plot(history['val_dice'], label='val pixel Dice')
axs[1].legend(); axs[1].set_title('Validation'); axs[1].set_xlabel('epoch')
plt.tight_layout(); plt.show()

## 10. Final evaluation on the best epoch
Compares **CLAHE on/off** and **flip TTA on/off** with a fine threshold sweep, so you pick the inference settings from data instead of by eye. Also reports COCO mask mAP for the winner.

In [ ]:
ck = torch.load(BEST_PATH, map_location=device, weights_only=False)
model.load_state_dict(ck['model'])
print(f"Loaded best epoch {ck['epoch'] + 1} (val F1 {ck['val_f1']:.4f})")

fine_thr = [round(x, 2) for x in np.arange(0.2, 0.96, 0.05)]
results = []
for clahe in (False, True):
    _, vl = make_val_loader(clahe)
    for tta in (False, True):
        rows = evaluate(model, vl, fine_thr, tta=tta)
        thr, r = best_row(rows)
        results.append(dict(clahe=clahe, tta=tta, thr=thr, **r))
        print(f'CLAHE={clahe!s:5} TTA={tta!s:5} | best thr {thr:.2f} | F1 {r["f1"]:.4f} '
              f'P {r["precision"]:.3f} R {r["recall"]:.3f} | Dice {r["dice"]:.4f} IoU {r["iou"]:.4f}')

best = max(results, key=lambda d: d['f1'])
print('\nWinner:', {k: (round(v, 4) if isinstance(v, float) else v) for k, v in best.items()})

_, vl = make_val_loader(best['clahe'])
mAP = coco_map(model, vl, tta=best['tta'])
print('\nCOCO mask metrics:', {k: round(v, 4) for k, v in mAP.items() if k in ('map', 'map_50', 'map_75', 'map_small', 'mar_100')})

INFERENCE_CFG = dict(threshold=best['thr'], clahe=best['clahe'], tta=best['tta'])
json.dump(INFERENCE_CFG, open(os.path.join(CHECKPOINT_DIR, 'inference_config.json'), 'w'))
print('Saved inference settings:', INFERENCE_CFG)

In [2]:
# ============================================================
# CLINDERMA FACIAL-SKIN PREPROCESSING
# Creates a binary mask: 255 = facial skin, 0 = excluded region
# ============================================================

import cv2
import numpy as np
import torch
import mediapipe as mp

from PIL import Image
from transformers import (
    AutoImageProcessor,
    AutoModelForSemanticSegmentation,
    SegformerImageProcessor
)

# Load the pretrained face parser once
FACE_MODEL_ID = "jonathandinu/face-parsing"

try:
    face_processor = AutoImageProcessor.from_pretrained(FACE_MODEL_ID)
except ValueError:
    face_processor = SegformerImageProcessor.from_pretrained(FACE_MODEL_ID)

face_parser = AutoModelForSemanticSegmentation.from_pretrained(
    FACE_MODEL_ID
)
face_parser.eval()

FACE_LABELS = {
    int(k): v.lower()
    for k, v in face_parser.config.id2label.items()
}

# MediaPipe face landmark detector
face_mesh_detector = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5
)


def generate_facial_skin_mask(image_rgb):
    """
    Input: RGB image as a NumPy array.
    Output: uint8 mask of the same height and width.
    """
    height, width = image_rgb.shape[:2]

    # Detect facial landmarks
    results = face_mesh_detector.process(image_rgb)

    if not results.multi_face_landmarks:
        raise ValueError(
            "No face detected. Check the image or use a face-visible image."
        )

    landmarks = results.multi_face_landmarks[0].landmark
    points = np.array([
        [
            int(np.clip(round(p.x * (width - 1)), 0, width - 1)),
            int(np.clip(round(p.y * (height - 1)), 0, height - 1))
        ]
        for p in landmarks
    ], dtype=np.int32)

    # Construct the face-oval boundary
    connections = mp.solutions.face_mesh.FACEMESH_FACE_OVAL
    neighbors = {}

    for a, b in connections:
        neighbors.setdefault(a, []).append(b)
        neighbors.setdefault(b, []).append(a)

    start = min(neighbors)
    path = [start]
    previous, current = None, start

    while True:
        candidates = [
            n for n in neighbors[current] if n != previous
        ]
        if not candidates:
            break

        nxt = candidates[0]
        if nxt == start:
            break

        path.append(nxt)
        previous, current = current, nxt

        if len(path) > len(neighbors):
            break

    face_oval = np.zeros((height, width), dtype=np.uint8)
    cv2.fillPoly(face_oval, [points[path]], 255)

    # Semantic face parsing
    pil_image = Image.fromarray(image_rgb)
    inputs = face_processor(images=pil_image, return_tensors="pt")

    with torch.no_grad():
        outputs = face_parser(**inputs)
        logits = torch.nn.functional.interpolate(
            outputs.logits,
            size=(height, width),
            mode="bilinear",
            align_corners=False
        )
        labels = logits.argmax(dim=1)[0].cpu().numpy()

    # Keep parsed skin; remove eyes, eyebrows, lips, hair, etc.
    skin_ids = [
        idx for idx, name in FACE_LABELS.items()
        if name == "skin"
    ]

    excluded_ids = [
        idx for idx, name in FACE_LABELS.items()
        if name in {
            "l_eye", "r_eye", "l_brow", "r_brow",
            "mouth", "u_lip", "l_lip", "hair",
            "hat", "eye_g"
        }
    ]

    skin_mask = np.isin(labels, skin_ids).astype(np.uint8) * 255

    if excluded_ids:
        excluded_mask = np.isin(labels, excluded_ids)
        skin_mask[excluded_mask] = 0

    # Restrict parsed skin to the detected facial oval
    skin_mask = cv2.bitwise_and(skin_mask, face_oval)

    # Remove small noise and fill small gaps
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (7, 7)
    )
    skin_mask = cv2.morphologyEx(
        skin_mask, cv2.MORPH_OPEN, kernel
    )
    skin_mask = cv2.morphologyEx(
        skin_mask, cv2.MORPH_CLOSE, kernel
    )

    # Keep the largest connected facial-skin region
    count, components, stats, _ = cv2.connectedComponentsWithStats(
        (skin_mask > 0).astype(np.uint8), 8
    )

    if count > 1:
        largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        skin_mask = np.where(components == largest, 255, 0).astype(
            np.uint8
        )

    return skin_mask


# Preview on an image already loaded as an RGB NumPy array:
# skin_mask = generate_facial_skin_mask(image_rgb)
# import matplotlib.pyplot as plt
# plt.figure(figsize=(10, 4))

# plt.subplot(1, 2, 1)
# plt.imshow(image_rgb)
# plt.title("Original image")
# plt.axis("off")
# plt.subplot(1, 2, 2)
# plt.imshow(skin_mask, cmap="gray")
# plt.title("Facial-skin mask")
# plt.axis("off")
# plt.show()

c:\Users\Dell\anaconda3\envs\clinderma-face\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 1172/1172 [00:00<00:00, 3303.25it/s]


## 11. Inference on a new image
Uses the winning settings (CLAHE yes/no, TTA yes/no, score threshold) from section 10. No more per-image brightness/contrast/negative hacks.

In [ ]:
def load_for_inference(path=BEST_PATH):
    ck = torch.load(path, map_location=device, weights_only=False)
    cfg = ck['config']
    m = build_model(pretrained=False, use_v2=cfg['use_v2'], anchor_sizes=cfg['anchor_sizes'],
                    num_classes=cfg['num_classes'])
    m.load_state_dict(ck['model'])
    return m.to(device).eval()

@torch.no_grad()
def predict_image(model, image_path, threshold=None, clahe=None, tta=None, show=True):
    cfg = json.load(open(os.path.join(CHECKPOINT_DIR, 'inference_config.json')))
    threshold = cfg['threshold'] if threshold is None else threshold
    clahe     = cfg['clahe']     if clahe is None else clahe
    tta       = cfg['tta']       if tta is None else tta

    rgb = np.array(Image.open(image_path).convert('RGB'))
    inp = apply_clahe(rgb) if clahe else rgb
    x = torch.from_numpy(inp).permute(2, 0, 1).float().div(255).to(device)
    with torch.autocast('cuda'):
        p = predict(model, [x], tta)[0]

    keep = p['scores'] >= threshold
    scores = p['scores'][keep].float().cpu().numpy()
    masks = (p['masks'][keep][:, 0] > 0.5).cpu().numpy()
    print(f'{len(scores)} pigmentation regions | scores:', np.round(scores, 2))

    if show:
        plt.figure(figsize=(12, 8)); plt.imshow(rgb)
        for m, sc in zip(masks, scores):
            ov = np.zeros((*m.shape, 4)); ov[m] = [1, 0, 0, 0.35]; plt.imshow(ov)
            ys, xs = np.where(m)
            if len(xs):
                plt.text(xs.min(), ys.min(), f'{sc:.2f}', color='white', fontsize=11,
                         bbox=dict(facecolor='black', alpha=0.7))
        plt.axis('off'); plt.title('Pigmentation detection'); plt.show()
    return scores, masks

infer_model = load_for_inference()
# predict_image(infer_model, '/content/your_image.jpg')